# UK Cross-Asset Market Monitor

This notebook tracks weekly moves across key UK and global market indicators, including gilt yields, UK equities, FX, commodities and global risk sentiment indicators.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import matplotlib.ticker as mtick

In [9]:
START_DATE = '2025-01-01'
END_DATE = None

ASSETS = {'FTSE100': '^FTSE', 'FTSE250': '^FTMC', 'GBP/USD': 'GBPUSD=X', 'EUR/GBP': 'EURGBP=X', 'Brent Crude': 'BZ=F', 'S&P500': '^GSPC', 'VIX': '^VIX', 'US 10Y Treasury Yield': '^TNX'}


# Market Data Download

This section downloads daily market data for the selected cross-asset indicators.

In [10]:
tickers = list(ASSETS.values())

market_data = yf.download(tickers, start=START_DATE, end=END_DATE, auto_adjust=True)

open_prices = market_data['Open'].rename(columns={v: k for k, v in ASSETS.items()})
close_prices = market_data['Close'].rename(columns={v: k for k, v in ASSETS.items()})
open_prices = open_prices.dropna(how='all')
close_prices = close_prices.dropna(how='all')

close_prices.tail()

[*********************100%***********************]  8 of 8 completed


Ticker,Brent Crude,EUR/GBP,GBP/USD,FTSE250,FTSE100,S&P500,US 10Y Treasury Yield,VIX
Date,,,,,,,,
2026-07-01,71.570000,0.86133,1.325065,23330.099609,10478.299805,7483.229980,4.475,16.59
2026-07-02,71.800003,0.85653,1.327933,23417.599609,10652.900391,7483.240234,4.485,16.15
2026-07-03,NaN,0.85610,1.333870,23538.800781,10679.000000,NaN,NaN,NaN
2026-07-06,71.989998,0.85649,1.335310,23504.199219,10651.799805,7537.430176,NaN,15.57
2026-07-07,73.760002,0.85456,1.337506,23404.529297,10695.610352,7484.850098,4.513,16.40


In [11]:
daily_returns = close_prices.pct_change().dropna(how='all')
daily_returns.tail()

Ticker,Brent Crude,EUR/GBP,GBP/USD,FTSE250,FTSE100,S&P500,US 10Y Treasury Yield,VIX
Date,,,,,,,,
2026-07-01,-0.018513,-0.000047,-0.000252,0.013757,-0.001791,-0.002151,0.012902,0.008511
2026-07-02,0.003214,-0.005573,0.002164,0.003751,0.016663,0.000001,0.002235,-0.026522
2026-07-03,NaN,-0.000502,0.004471,0.005176,0.002450,NaN,NaN,NaN
2026-07-06,NaN,0.000456,0.001079,-0.001470,-0.002547,NaN,NaN,NaN
2026-07-07,0.024587,-0.002253,0.001645,-0.004241,0.004113,-0.006976,NaN,0.053308


In [12]:
def create_weekly_summary_for_week(selected_date = None):
    if selected_date is None:
        latest_date = close_prices.dropna(how="all").index[-1]
    else:
        latest_date = pd.to_datetime(selected_date)

    latest_friday = latest_date - pd.Timedelta(days=(latest_date.weekday() - 4) % 7)
    week_start = latest_friday - pd.Timedelta(days=4)

    weekly_open_prices = open_prices.loc[(open_prices.index >= week_start) & (open_prices.index <= latest_friday)]

    weekly_close_prices = close_prices.loc[(close_prices.index >= week_start) & (close_prices.index <= latest_friday)]

    monday_open = weekly_open_prices.ffill().iloc[0]
    friday_close = weekly_close_prices.ffill().iloc[-1]

    weekly_change = friday_close - monday_open
    weekly_pct_change = (friday_close - monday_open) / monday_open

    weekly_summary = pd.DataFrame({ "Monday Open": monday_open, "Friday Close": friday_close, "Weekly Change": weekly_change, "Weekly % Change": weekly_pct_change })

    yield_assets = ['US 10Y Treasury Yield']
    weekly_summary['Change (bps)'] = np.nan
    for asset in yield_assets:
        weekly_summary.loc[asset, 'Change (bps)'] = weekly_summary.loc[asset, 'Weekly Change'] * 100
        weekly_summary.loc[asset, 'Weekly Change'] = np.nan
        weekly_summary.loc[asset, 'Weekly % Change'] = np.nan

    

    weekly_summary["Weekly % Change"] = weekly_summary["Weekly % Change"].map(lambda x:"" if pd.isna(x) else f"{x:.2%}")
    weekly_summary["Change (bps)"] = weekly_summary["Change (bps)"].map(lambda x:"" if pd.isna(x) else f"{x:.1f} bps")
    weekly_summary["Weekly Change"] = weekly_summary["Weekly Change"].map(lambda x:"" if pd.isna(x) else x)

    return weekly_summary

create_weekly_summary_for_week()

,Monday Open,Friday Close,Weekly Change,Weekly % Change,Change (bps)
Ticker,,,,,
Brent Crude,73.169998,71.800003,-1.369995,-1.87%,
EUR/GBP,0.862720,0.856100,-0.00662,-0.77%,
GBP/USD,1.319609,1.333870,0.014261,1.08%,
FTSE250,23147.000000,23538.800781,391.800781,1.69%,
FTSE100,10507.799805,10679.000000,171.200195,1.63%,
S&P500,7391.879883,7483.240234,91.360352,1.24%,
US 10Y Treasury Yield,4.378000,4.485000,,,10.7 bps
VIX,18.600000,16.150000,-2.450001,-13.17%,


In [13]:
create_weekly_summary_for_week("2026/07/07")

,Monday Open,Friday Close,Weekly Change,Weekly % Change,Change (bps)
Ticker,,,,,
Brent Crude,73.169998,71.800003,-1.369995,-1.87%,
EUR/GBP,0.862720,0.856100,-0.00662,-0.77%,
GBP/USD,1.319609,1.333870,0.014261,1.08%,
FTSE250,23147.000000,23538.800781,391.800781,1.69%,
FTSE100,10507.799805,10679.000000,171.200195,1.63%,
S&P500,7391.879883,7483.240234,91.360352,1.24%,
US 10Y Treasury Yield,4.378000,4.485000,,,10.7 bps
VIX,18.600000,16.150000,-2.450001,-13.17%,


In [14]:
create_weekly_summary_for_week("2026-07-01")

,Monday Open,Friday Close,Weekly Change,Weekly % Change,Change (bps)
Ticker,,,,,
Brent Crude,82.209999,71.989998,-10.220001,-12.43%,
EUR/GBP,0.867770,0.861330,-0.00644,-0.74%,
GBP/USD,1.320830,1.318791,-0.002038,-0.15%,
FTSE250,23201.000000,23147.199219,-53.800781,-0.23%,
FTSE100,10363.599609,10508.000000,144.400391,1.39%,
S&P500,7500.439941,7354.020020,-146.419922,-1.95%,
US 10Y Treasury Yield,4.495000,4.372000,,,-12.3 bps
VIX,17.480000,18.410000,0.93,5.32%,
